In [1]:
import pandas as pd

df = pd.read_csv("data/mosty_merged_2026.csv", sep=";")

In [2]:
df.columns

Index(['bridge_id', 'úsek_id', 'trieda_pk', 'číslo_pk', 'názov_mosta',
       'id_mosta', 'id_dc', 'správcovské_číslo', 'stavebno_technický_stav',
       'okres', 'dĺžka_premostenia', 'plocha_mosta', 'rok_postavenia',
       'zaťažovacia_trieda_stn', 'počet_otvorov', 'dĺžka_nosnej_konštrukcie',
       'volná_šírka_mosta', 'šírka_medzi_obrubami', 'pocha_mosnej_konštrukcie',
       'druh_konštrukcie', 'material', 'normána_zaťažiteľnost',
       'výhradná_zaťažiteľnosť', 'výnimočná_zaťažiteľnosť',
       'ojedinelá_zaťažiteľnosť', 'zaťažiteľnosť_na_nápravu',
       'spôsob_stanovenia', 'predmet_premostenia', 'longitude', 'latitude',
       'vyska', 'správca_dc_mostu', 'vlastník_dc_mostu',
       'úsekové_staničenie_začiatku_dc',
       'kilometrovníkové_staničenie_začiatku_dc',
       'kumulatívne_staničenie_začiatku_dc', 'poradie', 'smer_úseku_pk',
       'typ_úseku_pk', 'počiatočný_uzol_úseku_pk', 'koncový_uzol_úseku_pk',
       'správca_úseku_pk', 'vlastník_úseku_pk'],
      dtype='str

In [3]:
# Filter all bridges under Regionálna správa a údržba ciest Nitra
rscn_bridges = df[
    df['správca_úseku_pk'].str.contains('Regionálna správa a údržba ciest Nitra', na=False)
].copy()

# Verify row count and preview data
print(f"Total bridges found: {len(rscn_bridges)}")

Total bridges found: 536


In [4]:
rscn_bridges["material"].value_counts()

material
monolitický železobetón          285
prefabrikovaný železobetón       122
prefabrikovaný predpätý betón     60
oceľ -  plnostenná                32
kameň                             16
tehla                              9
monolitický predpätý betón         7
oceľ -  priehradová                2
oceľobetón                         2
Name: count, dtype: int64

In [5]:
rscn_bridges['dĺžka_nosnej_konštrukcie'].describe()

count     16.000000
mean      35.698125
std       46.171949
min        0.000000
25%        3.550000
50%       12.275000
75%       54.175000
max      164.100000
Name: dĺžka_nosnej_konštrukcie, dtype: float64

In [6]:
rscn_bridges['dĺžka_premostenia'].describe()

count    536.000000
mean      12.717597
std       25.917734
min        2.000000
25%        3.537500
50%        5.200000
75%        9.225000
max      266.700000
Name: dĺžka_premostenia, dtype: float64

In [7]:
rscn_bridges.columns

Index(['bridge_id', 'úsek_id', 'trieda_pk', 'číslo_pk', 'názov_mosta',
       'id_mosta', 'id_dc', 'správcovské_číslo', 'stavebno_technický_stav',
       'okres', 'dĺžka_premostenia', 'plocha_mosta', 'rok_postavenia',
       'zaťažovacia_trieda_stn', 'počet_otvorov', 'dĺžka_nosnej_konštrukcie',
       'volná_šírka_mosta', 'šírka_medzi_obrubami', 'pocha_mosnej_konštrukcie',
       'druh_konštrukcie', 'material', 'normána_zaťažiteľnost',
       'výhradná_zaťažiteľnosť', 'výnimočná_zaťažiteľnosť',
       'ojedinelá_zaťažiteľnosť', 'zaťažiteľnosť_na_nápravu',
       'spôsob_stanovenia', 'predmet_premostenia', 'longitude', 'latitude',
       'vyska', 'správca_dc_mostu', 'vlastník_dc_mostu',
       'úsekové_staničenie_začiatku_dc',
       'kilometrovníkové_staničenie_začiatku_dc',
       'kumulatívne_staničenie_začiatku_dc', 'poradie', 'smer_úseku_pk',
       'typ_úseku_pk', 'počiatočný_uzol_úseku_pk', 'koncový_uzol_úseku_pk',
       'správca_úseku_pk', 'vlastník_úseku_pk'],
      dtype='str

In [8]:
# Add cleaned numeric length to the DataFrame
rscn_bridges['dlzka_premostenia_num'] = pd.to_numeric(
    rscn_bridges['dĺžka_premostenia']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.replace(r'\xa0|\s+', '', regex=True),
    errors='coerce'
)

# Group by material and compute statistics
material_stats = (
    rscn_bridges.groupby('material')['dlzka_premostenia_num']
    .agg(
        pocet_mostov='count',
        min_dlzka='min',
        priemerna_dlzka='mean',
        max_dlzka='max',
        celkova_dlzka='sum'
    )
    .sort_values(by='pocet_mostov', ascending=False)
    .round(2)
)

material_stats

,pocet_mostov,min_dlzka,priemerna_dlzka,max_dlzka,celkova_dlzka
material,,,,,
monolitický železobetón,285,2.05,6.83,88.50,1946.98
prefabrikovaný železobetón,122,2.00,7.83,28.50,955.59
prefabrikovaný predpätý betón,60,8.15,39.93,266.70,2396.06
oceľ - plnostenná,32,2.30,12.71,72.00,406.85
kameň,16,2.40,4.88,15.20,78.05
tehla,9,2.05,2.83,4.20,25.45
monolitický predpätý betón,7,2.35,110.14,255.35,770.99
oceľ - priehradová,2,70.00,81.25,92.50,162.50
oceľobetón,2,3.00,36.08,69.15,72.15


In [13]:
print(material_stats)

                               pocet_mostov  min_dlzka  priemerna_dlzka  \
material                                                                  
monolitický železobetón                 285       2.05             6.83   
prefabrikovaný železobetón              122       2.00             7.83   
prefabrikovaný predpätý betón            60       8.15            39.93   
oceľ -  plnostenná                       32       2.30            12.71   
kameň                                    16       2.40             4.88   
tehla                                     9       2.05             2.83   
monolitický predpätý betón                7       2.35           110.14   
oceľ -  priehradová                       2      70.00            81.25   
oceľobetón                                2       3.00            36.08   

                               max_dlzka  celkova_dlzka  
material                                                 
monolitický železobetón            88.50        1946.98  


In [14]:
import numpy as np

# 1. KROK: Vyčistenie stĺpca 'počet_otvorov' (predpokladám, že sa volá 'počet_otvorov' alebo podobne)
# Ak sa v tvojom DF volá inak, uprav názov v zátvorkách
nazov_stlpca_otvory = 'počet_otvorov' if 'počet_otvorov' in rscn_bridges.columns else 'pocet_otvorov'

if nazov_stlpca_otvory in rscn_bridges.columns:
    rscn_bridges['pocet_poli_db'] = pd.to_numeric(
        rscn_bridges[nazov_stlpca_otvory]
        .astype(str)
        .str.replace(r'\xa0|\s+', '', regex=True),  # Odstráni neviditeľné medzery
        errors='coerce'
    )
else:
    # Ak stĺpec úplne chýba, vytvoríme ho prázdny pre účely ďalšieho kódu
    rscn_bridges['pocet_poli_db'] = np.nan

# Každý most musí mať minimálne 1 pole (ošetrenie núl alebo chýbajúcich hodnôt pre krátke mosty)
rscn_bridges['pocet_poli_db'] = rscn_bridges['pocet_poli_db'].fillna(1).clip(lower=1)

# 2. KROK: Inžinierske maximálne rozpätia jedného poľa podľa materiálu na SR (v metroch)
# Tieto hodnoty reprezentujú rozumné konštrukčné maximá pre jedno pole.
max_rozpatie = {
    'monolitický železobetón': 25.0,
    'prefabrikovaný železobetón': 21.0,     # Typické staré Vloššáky max 21m
    'prefabrikovaný predpätý betón': 35.0,  # I-nosníky (napr. I-73, I-96)
    'monolitický predpätý betón': 60.0,     # Letmá montáž a pod.
    'oceľ -  plnostenná': 50.0,
    'oceľ -  priehradová': 80.0,
    'oceľobetón': 40.0,
    'kameň': 10.0,                          # Kamenné klenby
    'tehla': 6.0
}

# Vytvoríme mapovanie pre výpočet odhadu
rscn_bridges['max_pole_pre_material'] = rscn_bridges['material'].map(max_rozpatie).fillna(20.0)

# Výpočet odhadovaného počtu polí (celková dĺžka / max dĺžka poľa, zaokrúhlené nahor)
# Ak má most dĺžku 30m a je z prefab. ŽB (max 21m), vyjde 30/21 = 1.42 -> po ceil() = 2 polia.
rscn_bridges['pocet_poli_odhad'] = np.ceil(rscn_bridges['dlzka_premostenia_num'] / rscn_bridges['max_pole_pre_material'])

# 3. KROK: Finálna syntéza - použijeme databázu, ale ak databáza hlási 1 pole a naša matika hovorí, 
# že konštrukčne to nie je možné, zoberieme odhad.
rscn_bridges['finalny_pocet_poli'] = np.where(
    (rscn_bridges['pocet_poli_db'] < rscn_bridges['pocet_poli_odhad']) & (rscn_bridges['pocet_poli_db'] == 1),
    rscn_bridges['pocet_poli_odhad'], 
    rscn_bridges['pocet_poli_db']
)

# SUMARIZÁCIA do tvojej pôvodnej tabuľky
material_stats_spans = (
    rscn_bridges.groupby('material')
    .agg(
        pocet_mostov=('dlzka_premostenia_num', 'count'),
        celkovy_pocet_poli=('finalny_pocet_poli', 'sum'),
        priemerna_dlzka=('dlzka_premostenia_num', 'mean'),
        celkova_dlzka=('dlzka_premostenia_num', 'sum')
    )
    .sort_values(by='pocet_mostov', ascending=False)
    .round(2)
)

print(material_stats_spans)
print(f"\nCelkový počet mostov v RSÚC Nitra: {rscn_bridges.shape[0]}")
print(f"Celkový počet mostných polí: {int(rscn_bridges['finalny_pocet_poli'].sum())}")

                               pocet_mostov  celkovy_pocet_poli  \
material                                                          
monolitický železobetón                 285               312.0   
prefabrikovaný železobetón              122               149.0   
prefabrikovaný predpätý betón            60               120.0   
oceľ -  plnostenná                       32                43.0   
kameň                                    16                19.0   
tehla                                     9                 9.0   
monolitický predpätý betón                7                28.0   
oceľ -  priehradová                       2                 3.0   
oceľobetón                                2                 4.0   

                               priemerna_dlzka  celkova_dlzka  
material                                                       
monolitický železobetón                   6.83        1946.98  
prefabrikovaný železobetón                7.83         955.59  
prefab

In [15]:
# 1. Vyčistenie a konverzia stĺpca počet_otvorov na číslo
rscn_bridges['pocet_otvorov_num'] = pd.to_numeric(
    rscn_bridges['počet_otvorov']
    .astype(str)
    .str.replace(r'\xa0|\s+', '', regex=True),
    errors='coerce'
)

# Ak je v databáze chýbajúca hodnota (NaN), pre existujúci most je to minimálne 1 otvor
rscn_bridges['pocet_otvorov_num'] = rscn_bridges['pocet_otvorov_num'].fillna(1).clip(lower=1)

# 2. Agregácia podľa materiálu
material_stats_otvory = (
    rscn_bridges.groupby('material')
    .agg(
        pocet_mostov=('dlzka_premostenia_num', 'count'),
        celkovy_pocet_poli_otvorov=('pocet_otvorov_num', 'sum'),
        priemerny_pocet_poli_na_most=('pocet_otvorov_num', 'mean'),
        priemerna_dlzka_mosta=('dlzka_premostenia_num', 'mean'),
        celkova_dlzka=('dlzka_premostenia_num', 'sum')
    )
    .sort_values(by='pocet_mostov', ascending=False)
    .round(2)
)

# 3. Priemerná dĺžka jedného poľa (rozpätie)
material_stats_otvory['priemerna_dlzka_jedneho_polia'] = (
    material_stats_otvory['celkova_dlzka'] / material_stats_otvory['celkovy_pocet_poli_otvorov']
).round(2)

material_stats_otvory

,pocet_mostov,celkovy_pocet_poli_otvorov,priemerny_pocet_poli_na_most,priemerna_dlzka_mosta,celkova_dlzka,priemerna_dlzka_jedneho_polia
material,,,,,,
monolitický železobetón,285,305.0,1.07,6.83,1946.98,6.38
prefabrikovaný železobetón,122,149.0,1.22,7.83,955.59,6.41
prefabrikovaný predpätý betón,60,120.0,2.00,39.93,2396.06,19.97
oceľ - plnostenná,32,43.0,1.34,12.71,406.85,9.46
kameň,16,19.0,1.19,4.88,78.05,4.11
tehla,9,9.0,1.00,2.83,25.45,2.83
monolitický predpätý betón,7,28.0,4.00,110.14,770.99,27.54
oceľ - priehradová,2,3.0,1.50,81.25,162.50,54.17
oceľobetón,2,4.0,2.00,36.08,72.15,18.04


In [17]:
material_stats_otvory['celkovy_pocet_poli_otvorov'].sum()

np.float64(680.0)